# Run RAG Query

This notebook is to run the RAG query. 

In [10]:
from pathlib import Path
from typing import List

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader

# Load documents as PDFs
from langchain_community.document_loaders import PyPDFLoader

# Chunk documents
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embedding Model
from langchain_community.embeddings import HuggingFaceEmbeddings

# Create vector DB
from langchain_community.vectorstores import FAISS

print("Imports loaded.")


Imports loaded.


# Reload Vector DB 

In [11]:
# Embedding Model. Using free tier AI
# from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 15309.31it/s]


In [12]:
save_path = "fixed_rag_faiss_index"

vector_db = FAISS.load_local(
    save_path,
    embeddings,
    allow_dangerous_deserialization=True
)

print("Vector DB reloaded from local folder.")

Vector DB reloaded from local folder.


In [13]:
# Retriever
retriever = vector_db.as_retriever(search_kwargs={"k": 4})

In [14]:
# Load local Hugging Face model for answer generation
# TinyLlama is a causal model, so we must format the prompt as chat and decode ONLY new tokens.

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_id)

# Use CPU-safe loading. If you have a GPU, you can change this later.
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float32,
    device_map=None
)

model.eval()
print("LLM loaded.")


Loading weights: 100%|██████████| 201/201 [00:03<00:00, 52.75it/s]


LLM loaded.


In [15]:
def llm(prompt: str, max_new_tokens: int = 120) -> str:
    """
    Generate an answer using TinyLlama.
    Important fix: decode only the newly generated tokens, not the full prompt.
    """

    messages = [
        {
            "role": "system",
            "content": (
                "You are a careful RAG assistant. Answer only from the provided context. "
                "Use citations exactly as provided, such as [1] or [1, p. 3]. "
                "If the context does not answer the question, say that the documents do not contain enough information."
            )
        },
        {"role": "user", "content": prompt}
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    # CRITICAL FIX: remove the prompt tokens from the output
    new_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True)

    return answer.strip()


In [16]:
def retrieve_chunks(query: str, k: int = 4):
    """Retrieve top-k chunks from the FAISS vector database."""
    return vector_db.similarity_search(query, k=k)


# Quick retrieval test
test_chunks = retrieve_chunks("What is copyright?", k=4)
print(f"Retrieved {len(test_chunks)} chunks")

for i, doc in enumerate(test_chunks, start=1):
    print(f"\nChunk {i}")
    print("Source:", doc.metadata.get("source_file", "Unknown file"))
    print("Page:", doc.metadata.get("page", "Unknown page"))
    print("Citation:", doc.metadata.get("ieee_citation", "Unknown citation"))
    print("Preview:", doc.page_content[:250].replace("\n", " "))


Retrieved 4 chunks

Chunk 1
Source: 3614407.3643696.pdf
Page: 8
Citation: Katherine Lee, A. Feder Cooper, James Grimmelmann, "Talkin' 'Bout AI Generation," Proceedings of the Symposium on Computer Science and Law, ACM, 2024, doi: 10.1145/3614407.3643696.
Preview: and to noncommercial use (e.g., illustrating an academic article on generative AI). Some outputs will be put to favored purposes like education and news reporting, while other outputs will be put to run-of-the-mill entertainment purposes.75 Factor Tw

Chunk 2
Source: 3614407.3643696.pdf
Page: 5
Citation: Katherine Lee, A. Feder Cooper, James Grimmelmann, "Talkin' 'Bout AI Generation," Proceedings of the Symposium on Computer Science and Law, ACM, 2024, doi: 10.1145/3614407.3643696.
Preview: right law to the generative-AI supply chain. We address issues of rights (Section 3.2), infringement (Sections 3.3 & 3.4), and fair use (Section 3.5). We defer discussion of safe harbors, licenses, para- copyright liability, and remedies to

In [17]:
def rag_query(question: str, k: int = 4) -> dict:
    """
    RAG pipeline focused only on retrieval + answer generation.
    Assumes citations already exist in the FAISS metadata.
    """
    try:
        retrieved_docs = retrieve_chunks(question, k=k)

        if not retrieved_docs:
            return {
                "question": question,
                "answer": "I could not retrieve any relevant document chunks.",
                "references": [],
                "retrieved_docs": []
            }

        context_blocks = []
        references = []
        seen_citations = set()

        for i, doc in enumerate(retrieved_docs, start=1):
            page = doc.metadata.get("page", "Unknown page")
            source_file = doc.metadata.get("source_file", "Unknown file")
            citation = doc.metadata.get("ieee_citation", "Unknown citation")

            context_blocks.append(
                f"""
[{i}]
File: {source_file}
Page: {page}
Citation: {citation}
Text:
{doc.page_content}
"""
            )

            if citation not in seen_citations:
                references.append(f"[{i}] {citation}")
                seen_citations.add(citation)

        context = "\n---\n".join(context_blocks)

        prompt = f"""
Use ONLY the context below to answer the question.

Question: {question}

Context:
{context}

Instructions:
- Give a direct answer.
- Use inline citations like [1] or [1, p. 3].
- Do not repeat the full prompt.
- Do not make up information outside the context.
- If the answer is not in the context, say the documents do not contain enough information.
- End with a References section using the citations provided in the context.

Answer:
"""

        answer = llm(prompt)

        # If the small model forgets references, append them deterministically.
        if "References" not in answer:
            answer += "\n\nReferences:\n" + "\n".join(references)

        return {
            "question": question,
            "answer": answer,
            "references": references,
            "retrieved_docs": retrieved_docs
        }

    except Exception as e:
        print(f"Error: {e}")
        return {"error": str(e)}


In [18]:
# Test the RAG pipeline
result = rag_query("What is the impact of LLMs on copyright?", k=2)

if "error" in result:
    print("RAG failed:", result["error"])
else:
    print("Question:", result["question"])
    print("\nAnswer:\n")
    print(result["answer"])


[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: What is the impact of LLMs on copyright?

Answer:

The impact of LLMs on copyright is significant, as they can have a significant impact on the nature of the copyrighted work. Some training data will be informational, while others will be expressive. Most training data will have been "published" within the meaning of copyright law, whereas some may be "unpublished" within the meaning of copyright law. A very small fraction of training data may be "unpublished" within the meaning of copyright law, which would not be available as training data at all. This factor depends on the model in question

References:
[1] Katherine Lee, A. Feder Cooper, James Grimmelmann, "Talkin' 'Bout AI Generation," Proceedings of the Symposium on Computer Science and Law, ACM, 2024, doi: 10.1145/3614407.3643696.
[2] Rui-Jie Yew, "Break It 'Til You Make It: An Exploration of the Ramifications of Copyright Liability Under a Pre-training Paradigm of AI Development," Proceedings of the Symposium on Compu